In [1]:
import pickle
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import json
import numpy as np
import geneformer
from geneformer import TranscriptomeTokenizer
import scanpy as sc
import anndata as an
from datasets import Dataset, load_from_disk
from scipy.spatial.distance import cdist
import torch
from tqdm import tqdm
import plotly.graph_objects as go
import gc
from importlib import reload
import umap

# local functions
sys.path.append("../../utils/")
import geneformer_utils as ut

In [2]:
import plotly.io as pio
pio.renderers.default = 'notebook'  # or 'inline' or 'notebook_connected'

# Load gene mappping

In [3]:
fpath = "/home/cstansbu/git_repositories/hematokytos/resources/gene_names.tsv.gz"
gdf = pd.read_csv(fpath, sep='\t')
print(gdf.shape)

id2name = dict(zip(gdf['Gene stable ID'].values, gdf['Gene name'].values))

gdf.head()

(73466, 8)


/tmp/ipykernel_1331600/1886190844.py:2: DtypeWarning:

Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.



,Gene stable ID,Gene description,Gene start (bp),Gene end (bp),Chromosome/scaffold name,Gene type,Gene % GC content,Gene name
0,ENSG00000210049,mitochondrially encoded tRNA-Phe (UUU/C) [Sour...,577,647.0,MT,Mt_tRNA,40.85,MT-TF
1,ENSG00000211459,mitochondrially encoded 12S rRNA [Source:HGNC ...,648,1601.0,MT,Mt_rRNA,45.49,MT-RNR1
2,ENSG00000210077,mitochondrially encoded tRNA-Val (GUN) [Source...,1602,1670.0,MT,Mt_tRNA,42.03,MT-TV
3,ENSG00000210082,mitochondrially encoded 16S rRNA [Source:HGNC ...,1671,3229.0,MT,Mt_rRNA,42.81,MT-RNR2
4,ENSG00000209082,mitochondrially encoded tRNA-Leu (UUA/G) 1 [So...,3230,3304.0,MT,Mt_tRNA,38.67,MT-TL1


In [4]:
gdf[gdf['Gene name'] == 'GATA2']

,Gene stable ID,Gene description,Gene start (bp),Gene end (bp),Chromosome/scaffold name,Gene type,Gene % GC content,Gene name
72889,ENSG00000179348,GATA binding protein 2 [Source:HGNC Symbol;Acc...,128479427,128493201.0,3.0,protein_coding,60.46,GATA2


# geneformer resources

In [5]:
model_str = '95m'
params_file = "../../resources/geneformer_params.yaml"

with open(params_file, 'r') as file:
    params = yaml.safe_load(file)

model = params['models'][model_str]
print(json.dumps(model, indent=2))

with open(model['token_dictionary_file'], 'rb') as f:  
    tokens = pickle.load(f)

tokens_r = {v: k for k, v in tokens.items()}
print(f"{len(tokens)=}")

with open(model['gene_mapping_file'], 'rb') as f:  
    gene_map = pickle.load(f)

print(f"{len(gene_map)=}")

{
  "model_path": "/nfs/turbo/umms-indikar/shared/projects/foundation_models/geneformer/Geneformer/gf-12L-95M-i4096/",
  "gene_median": "/nfs/turbo/umms-indikar/shared/projects/foundation_models/geneformer/Geneformer/geneformer/gene_median_dictionary_gc95M.pkl",
  "token_dictionary_file": "/nfs/turbo/umms-indikar/shared/projects/foundation_models/geneformer/Geneformer/geneformer/token_dictionary_gc95M.pkl",
  "gene_mapping_file": "/nfs/turbo/umms-indikar/shared/projects/foundation_models/geneformer/Geneformer/geneformer/ensembl_mapping_dict_gc95M.pkl",
  "model_input_size": 4096,
  "special_token": true
}
len(tokens)=20275
len(gene_map)=173697


In [6]:
tf_recipe = ['GATA2', 'GFI1B', 'FOS', 'STAT5A', 'REL']

tf_df = []
for tf in tf_recipe:
    ens_id = gene_map[tf]
    token_id = tokens[ens_id]
    tf_df.append((tf, ens_id, token_id))

tf_df = pd.DataFrame(tf_df, columns=["TF", "ensembl_ID", "token_ID"])

tf_map = dict(zip(tf_df['ensembl_ID'].values, tf_df['TF'].values))
print(tf_df.to_string(index=False))

    TF      ensembl_ID  token_ID
 GATA2 ENSG00000179348     14205
 GFI1B ENSG00000165702     11475
   FOS ENSG00000170345     12558
STAT5A ENSG00000126561      5761
   REL ENSG00000162924     10694


In [7]:
tf_map

{'ENSG00000179348': 'GATA2',
 'ENSG00000165702': 'GFI1B',
 'ENSG00000170345': 'FOS',
 'ENSG00000126561': 'STAT5A',
 'ENSG00000162924': 'REL'}

# Example plots

In [8]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

# Our Data

In [ ]:
%%time
fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/geneformer/pseudotime.dataset"

data = load_from_disk(fpath)
print(f"{data.shape=}")

df = data.to_pandas()
df.head()

In [ ]:
n_cells = 10
n_genes = 10

input_tfs = tf_df['token_ID'].to_list()

results = []

for idx, row in tqdm(df.head(n_cells).iterrows(), total=n_cells, desc="Processing cells"):
    unperturbed = list(row['input_ids'])[1:n_genes+1]
    perturbed = (input_tfs + unperturbed)[:len(unperturbed)]

    tmp = pd.DataFrame({
        'unperturbed_token_id': [tokens_r[i] for i in unperturbed],
        'perturbed_token_id': [tokens_r[i] for i in perturbed],
    })

    tmp['unperturbed_gene'] = tmp['unperturbed_token_id'].map(id2name)
    tmp['perturbed_gene'] = tmp['perturbed_token_id'].map(id2name)
    tmp['perturbed_gene'] = np.where(tmp['perturbed_gene'].isna(), tmp['perturbed_token_id'].map(tf_map), tmp['perturbed_gene'])
    tmp = tmp.drop(columns=['unperturbed_token_id', 'perturbed_token_id'])
    tmp['unperturbed_gene_rank'] = range(n_genes)
    tmp['perturbed_gene_rank'] = range(n_genes)
    tmp['cell_id'] = f'cell_{idx}'
    tmp['cluster_str'] = row['cluster_str']
    results.append(tmp)


results = pd.concat(results)
print(results.head().to_string())

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Average ranks per cluster
avg_df = (
    results
    .groupby(['cluster_str', 'perturbed_gene', 'unperturbed_gene'])
    .mean(numeric_only=True)
    .reset_index()
)

# One Sankey plot per cluster
for cluster, df in avg_df.groupby('cluster_str'):
    genes = pd.unique(df[['perturbed_gene', 'unperturbed_gene']].values.ravel())
    gene_to_idx = {g: i for i, g in enumerate(genes)}

    fig = go.Figure(go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=list(gene_to_idx.keys())
        ),
        link=dict(
            source=df['perturbed_gene'].map(gene_to_idx),
            target=df['unperturbed_gene'].map(gene_to_idx),
            value=[1] * len(df)
        )
    ))

    fig.update_layout(title_text=f"Cluster: {cluster}", font_size=10)
    fig.show()


In [ ]:
break

In [ ]:
# Rank genes by order in each cell
results['unperturbed_rank'] = results.groupby('cell_id').cumcount() + 1
results['perturbed_rank'] = results.groupby('cell_id').cumcount() + 1

# Invert rank so that high ranks = high values
results['unperturbed_value'] = 1 / results['unperturbed_rank']
results['perturbed_value'] = 1 / results['perturbed_rank']

# Make unique node list
all_nodes = pd.unique(results['cell_id'].tolist() + 
                      results['unperturbed_gene'].tolist() + 
                      results['perturbed_gene'].tolist())
node_indices = {name: idx for idx, name in enumerate(all_nodes)}

# Link construction
sources = []
targets = []
values = []

# cell → unperturbed
for _, row in results.iterrows():
    sources.append(node_indices[row['cell_id']])
    targets.append(node_indices[row['unperturbed_gene']])
    values.append(row['unperturbed_value'])

# unperturbed → perturbed
for _, row in results.iterrows():
    sources.append(node_indices[row['unperturbed_gene']])
    targets.append(node_indices[row['perturbed_gene']])
    values.append(row['perturbed_value'])

# Sankey plot
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=15,
        line=dict(color="black", width=0.5),
        label=list(all_nodes)
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
)])

fig.update_layout(title_text="Cell-Centric Gene Rank Perturbation", font_size=10)
fig.show()

In [ ]:
# import pandas as pd
# import plotly.graph_objects as go
# import plotly.io as pio
# pio.renderers.default = 'notebook'  # or 'inline' or 'notebook_connected'

# # Example data structure: Source → Intermediate → Target
# df = pd.DataFrame({
#     'source': ['A', 'B', 'C', 'A', 'C', 'D'],
#     'intermediate': ['X1', 'X1', 'X2', 'X2', 'X3', 'X3'],
#     'target': ['Z1', 'Z2', 'Z1', 'Z3', 'Z2', 'Z3'],
#     'value': [5, 3, 6, 4, 2, 7]
# })

# # Create unique labels
# labels = list(pd.unique(df[['source', 'intermediate', 'target']].values.ravel()))
# label_to_index = {label: i for i, label in enumerate(labels)}

# # Define nodes
# source = df['source'].map(label_to_index)
# intermediate = df['intermediate'].map(label_to_index)
# target = df['target'].map(label_to_index)

# # Create sankey links: source → intermediate and intermediate → target
# links = pd.concat([
#     pd.DataFrame({'source': source, 'target': intermediate, 'value': df['value']}),
#     pd.DataFrame({'source': intermediate, 'target': target, 'value': df['value']})
# ])

# # Optional color mapping (based on label groups)
# color_map = {
#     'A': 'rgba(255,0,0,0.6)', 'B': 'rgba(0,255,0,0.6)', 'C': 'rgba(0,0,255,0.6)',
#     'D': 'rgba(255,0,255,0.6)', 'X1': 'gray', 'X2': 'gray', 'X3': 'gray',
#     'Z1': 'black', 'Z2': 'black', 'Z3': 'black'
# }
# link_colors = [color_map.get(labels[s], 'lightgray') for s in links['source']]

# # Plot
# fig = go.Figure(data=[go.Sankey(
#     node=dict(
#         pad=15,
#         thickness=20,
#         line=dict(color="black", width=0.5),
#         label=labels
#     ),
#     link=dict(
#         source=links['source'],
#         target=links['target'],
#         value=links['value'],
#         color=link_colors
#     )
# )])
# fig.update_layout(title_text="Multi-stage Sankey Diagram", font_size=10)
# fig.show()
